# 🛡️ NB3 — Méthodes Robustes au Bruit de Labels
**Projet 1 : Data-Centric Deep Learning for Noisy Labels**

Ce notebook implémente et entraîne **3 méthodes robustes** :
1. **Label Smoothing** — régularisation via soft labels
2. **Sample Reweighting** — pondération inverse à la loss
3. **Small-Loss Selection** — filtrage actif des samples bruités

Chaque méthode tourne sur les **mêmes 5 configurations** que NB2 → comparaison équitable.

⏱️ **Durée estimée : ~4-5h sur Kaggle T4 GPU**

⚠️ **Prérequis : avoir exécuté NB1 et NB2**

In [ ]:
import random, os, pickle, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── Seeds ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

CLASSES   = ['airplane','automobile','bird','cat','deer',
             'dog','frog','horse','ship','truck']
N_CLASSES = 10
NOISE_RATES = [0.0, 0.2, 0.4, 0.6]
CONFIGS = [
    (0.0, 'symmetric'),
    (0.2, 'symmetric'),
    (0.4, 'symmetric'),
    (0.6, 'symmetric'),
    (0.4, 'asymmetric'),
]

print(f'✅ Setup OK | Device : {device}')
if torch.cuda.is_available():
    print(f'   GPU : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── CIFAR-10 ──────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
clean_trainset = torchvision.datasets.CIFAR10(
    './data', train=True,  download=True, transform=transform_train)
testset        = torchvision.datasets.CIFAR10(
    './data', train=False, download=True, transform=transform_test)
test_loader    = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)
print(f'✅ CIFAR-10 chargé')


# ── NoisyDataset ──────────────────────────────────────────────
class NoisyDataset(Dataset):
    ASYM_MAP = {0:0,1:2,2:0,3:5,4:7,5:3,6:6,7:4,8:8,9:1}
    def __init__(self, dataset, noise_rate=0.0,
                 noise_type='symmetric', seed=42):
        self.dataset      = dataset
        self.noise_rate   = noise_rate
        self.clean_labels = np.array(dataset.targets)
        self.noisy_labels = self.clean_labels.copy()
        if noise_rate > 0:
            rng = np.random.RandomState(seed)
            idx = rng.choice(len(self.clean_labels),
                             int(len(self.clean_labels)*noise_rate), replace=False)
            for i in idx:
                orig = self.clean_labels[i]
                if noise_type == 'symmetric':
                    self.noisy_labels[i] = rng.choice([c for c in range(10) if c!=orig])
                else:
                    t = self.ASYM_MAP[orig]
                    if t != orig: self.noisy_labels[i] = t
        self.noise_mask        = self.noisy_labels != self.clean_labels
        self.actual_noise_rate = self.noise_mask.mean()
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        return img, self.noisy_labels[idx], idx


# ── ResNet-18 ─────────────────────────────────────────────────
def get_resnet18():
    m = models.resnet18(pretrained=False)
    m.conv1  = nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, N_CLASSES)
    return m.to(device)


# ── Helpers ───────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for batch in loader:
        imgs, lbls = batch[0].to(device), batch[1].to(device)
        correct += model(imgs).argmax(1).eq(lbls).sum().item()
        total   += lbls.size(0)
    return 100.*correct/total

def make_loader(ds, batch_size=256):
    return DataLoader(ds, batch_size=batch_size,
                      shuffle=True, num_workers=2, pin_memory=True)

def make_optimizer_scheduler(model, lr=0.1, n_epochs=30):
    opt  = torch.optim.SGD(model.parameters(), lr=lr,
                            momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    return opt, sched

print('✅ Classes et helpers définis')

---
## 🔵 Méthode 1 — Label Smoothing

### Principe
Au lieu d'entraîner avec des **hard labels** (0 ou 1), on utilise des **soft labels** qui distribuent une petite probabilité ε à toutes les classes :

$$\tilde{y}_k = \begin{cases} 1 - \varepsilon + \varepsilon/K & \text{si } k = y \\ \varepsilon / K & \text{sinon} \end{cases}$$

### Justification
Si un label est bruité, le gradient incitant le modèle à être très confiant dans cette direction est **atténué**. Le modèle n'est plus pénalisé pour ne pas être à 100% certain d'un label potentiellement faux. Cela agit comme une **régularisation implicite** contre la mémorisation.

**Paramètre ε = 0.1** : valeur standard (Szegedy et al. 2016, Inception v3). Trop petit (< 0.05) : effet négligeable. Trop grand (> 0.3) : le modèle perd sa capacité de discrimination.

In [ ]:
class LabelSmoothingLoss(nn.Module):
    """
    Cross-Entropy avec Label Smoothing.
    ε = smoothing : probabilité redistribuée sur les fausses classes.
    """
    def __init__(self, num_classes=10, smoothing=0.1):
        super().__init__()
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing
        self.K          = num_classes

    def forward(self, pred, target):
        log_prob    = F.log_softmax(pred, dim=1)
        smooth_val  = self.smoothing / (self.K - 1)
        one_hot     = torch.zeros_like(pred).scatter_(1, target.unsqueeze(1), 1)
        soft_labels = one_hot * self.confidence + (1 - one_hot) * smooth_val
        return -(soft_labels * log_prob).sum(dim=1).mean()


def run_label_smoothing(noise_rate, noise_type,
                        n_epochs=30, smoothing=0.1):
    torch.manual_seed(SEED); np.random.seed(SEED)
    ds        = NoisyDataset(clean_trainset, noise_rate, noise_type)
    loader    = make_loader(ds)
    model     = get_resnet18()
    opt, sched = make_optimizer_scheduler(model, n_epochs=n_epochs)
    criterion = LabelSmoothingLoss(N_CLASSES, smoothing)
    history   = {'train_loss':[], 'train_acc':[], 'test_acc':[]}

    for ep in range(n_epochs):
        model.train(); loss_sum = correct = total = 0
        for imgs, lbls, _ in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward(); opt.step()
            loss_sum += loss.item()
            correct  += out.argmax(1).eq(lbls).sum().item()
            total    += lbls.size(0)
        sched.step()
        te_acc = evaluate(model, test_loader)
        history['train_loss'].append(loss_sum/len(loader))
        history['train_acc'].append(100.*correct/total)
        history['test_acc'].append(te_acc)
        if (ep+1) % 10 == 0:
            print(f'  Ep {ep+1:3d} | '
                  f'Loss {loss_sum/len(loader):.3f} | '
                  f'Train {100.*correct/total:.1f}% | '
                  f'Test {te_acc:.1f}%')

    history['best_test_acc'] = max(history['test_acc'])
    return history


print('✅ Label Smoothing défini')

# ── Vérification rapide ───────────────────────────────────────
p = torch.randn(8,10); t = torch.randint(0,10,(8,))
ls_loss = LabelSmoothingLoss(10, 0.1)(p, t)
ce_loss = nn.CrossEntropyLoss()(p, t)
print(f'   CE loss    : {ce_loss.item():.4f}')
print(f'   LS loss    : {ls_loss.item():.4f}')
print(f'   Différence : {abs(ce_loss.item()-ls_loss.item()):.4f} (attendu ~0.1)')

In [ ]:
results_smoothing = {}
t0 = time.time()

print('🚀 LABEL SMOOTHING (ε=0.1)')
print('=' * 55)

for nr, nt in CONFIGS:
    key = f'{nt}_{nr}'
    print(f'\n▶ {nt:<11} | ε = {nr}')
    hist = run_label_smoothing(nr, nt, n_epochs=30)
    results_smoothing[key] = hist
    print(f'   ✅ Best test acc : {hist["best_test_acc"]:.1f}%')

print(f'\n✅ Label Smoothing terminé en {(time.time()-t0)/60:.1f} min')
with open('results_smoothing.pkl', 'wb') as f:
    pickle.dump(results_smoothing, f)
print('💾 results_smoothing.pkl sauvegardé')

---
## 🟢 Méthode 2 — Sample Reweighting

### Principe
Les exemples bruités ont tendance à avoir une **loss élevée** car le modèle ne parvient pas à fitter des exemples incohérents (image de chat avec label "chien"). On attribue des poids **inversement proportionnels** à la loss :

$$w_i = \exp(-\mathcal{L}_i / T)$$

Un sample avec haute loss (probablement bruité) reçoit un faible poids → moins d'influence sur le gradient.

### Justification
Basé sur l'observation de **Arpit et al. (2017)** : les réseaux de neurones apprennent d'abord les exemples "propres" (patterns simples, faible loss) avant de mémoriser le bruit. En période d'entraînement intermédiaire, la loss est donc un signal discriminant fiable.

**Limite** : ce signal devient moins fiable à mesure que le modèle mémorise → on recalcule les poids tous les 5 epochs.

In [ ]:
@torch.no_grad()
def compute_weights(model, loader, n_samples, temperature=2.0):
    """
    Calcule w_i = exp(-L_i / T) pour chaque sample.
    temperature : contrôle le contraste des poids.
      Petit T → très contrasté (agressif)
      Grand T → poids proches (doux)
    T=2.0 : bon compromis (pas trop agressif pour ne pas
    sous-pondérer les hard but clean examples).
    """
    model.eval()
    losses   = torch.zeros(n_samples, device=device)
    crit_none = nn.CrossEntropyLoss(reduction='none')
    for imgs, lbls, idx in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        losses[idx] = crit_none(model(imgs), lbls)
    w = torch.exp(-losses / temperature)
    return w / w.mean()   # normaliser pour conserver l'échelle du LR


def run_reweighting(noise_rate, noise_type, n_epochs=30):
    torch.manual_seed(SEED); np.random.seed(SEED)
    ds        = NoisyDataset(clean_trainset, noise_rate, noise_type)
    loader    = make_loader(ds)
    model     = get_resnet18()
    opt, sched = make_optimizer_scheduler(model, n_epochs=n_epochs)
    crit_none  = nn.CrossEntropyLoss(reduction='none')
    history    = {'train_loss':[], 'train_acc':[], 'test_acc':[]}
    weights    = torch.ones(len(ds), device=device)  # poids initiaux uniformes

    for ep in range(n_epochs):
        # Recalculer les poids tous les 5 epochs
        if ep % 5 == 0:
            weights = compute_weights(model, loader, len(ds))

        model.train(); loss_sum = correct = total = 0
        for imgs, lbls, idx in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            w = weights[idx].detach()
            opt.zero_grad()
            out  = model(imgs)
            loss = (crit_none(out, lbls) * w).mean()
            loss.backward(); opt.step()
            loss_sum += loss.item()
            correct  += out.argmax(1).eq(lbls).sum().item()
            total    += lbls.size(0)
        sched.step()
        te_acc = evaluate(model, test_loader)
        history['train_loss'].append(loss_sum/len(loader))
        history['train_acc'].append(100.*correct/total)
        history['test_acc'].append(te_acc)
        if (ep+1) % 10 == 0:
            w_stats = weights.cpu().numpy()
            print(f'  Ep {ep+1:3d} | '
                  f'Loss {loss_sum/len(loader):.3f} | '
                  f'Train {100.*correct/total:.1f}% | '
                  f'Test {te_acc:.1f}% | '
                  f'w∈[{w_stats.min():.2f},{w_stats.max():.2f}]')

    history['best_test_acc'] = max(history['test_acc'])
    return history

print('✅ Sample Reweighting défini')

In [ ]:
results_reweighting = {}
t0 = time.time()

print('🚀 SAMPLE REWEIGHTING')
print('=' * 55)

for nr, nt in CONFIGS:
    key = f'{nt}_{nr}'
    print(f'\n▶ {nt:<11} | ε = {nr}')
    hist = run_reweighting(nr, nt, n_epochs=30)
    results_reweighting[key] = hist
    print(f'   ✅ Best test acc : {hist["best_test_acc"]:.1f}%')

print(f'\n✅ Reweighting terminé en {(time.time()-t0)/60:.1f} min')
with open('results_reweighting.pkl', 'wb') as f:
    pickle.dump(results_reweighting, f)
print('💾 results_reweighting.pkl sauvegardé')

---
## 🟣 Méthode 3 — Small-Loss Selection

### Principe
À chaque epoch, on **filtre activement** les samples suspects : on calcule la loss de tous les samples, on trie par ordre croissant, et on entraîne **uniquement** sur les R(t)% avec la plus petite loss.

R(t) décroît progressivement de 100% vers `1 - noise_rate` :
$$R(t) = 1 - \text{noise\_rate} \cdot \min\left(\frac{t}{T_{warmup}}, 1\right)$$

### Inspiration
Han et al. (2018) **"Co-Teaching"** — NIPS 2018. La version complète utilise deux réseaux qui s'échangent mutuellement leurs samples propres. Nous implémentons la version **single-network** (plus simple, légèrement moins robuste mais suffisant pour ce projet).

### Avantage vs Reweighting
Le reweighting **réduit** l'influence des bruités, la small-loss selection les **exclut complètement**. Plus efficace à haut noise rate (> 40%) mais risque de perdre des samples propres difficiles.

In [ ]:
@torch.no_grad()
def select_clean_samples(model, loader, dataset,
                          keep_ratio, batch_size=256):
    """
    Retourne un DataLoader contenant uniquement les keep_ratio%
    d'exemples avec la plus petite loss (les plus probablement propres).
    """
    model.eval()
    crit_none  = nn.CrossEntropyLoss(reduction='none')
    all_losses = []
    all_idx    = []

    for imgs, lbls, idx in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        all_losses.extend(crit_none(model(imgs), lbls).cpu().numpy())
        all_idx.extend(idx.numpy())

    all_losses = np.array(all_losses)
    all_idx    = np.array(all_idx)

    n_keep   = max(batch_size, int(len(all_losses) * keep_ratio))
    selected = all_idx[np.argsort(all_losses)[:n_keep]]  # indices petite loss

    return DataLoader(Subset(dataset, selected),
                      batch_size=batch_size,
                      shuffle=True, num_workers=2, pin_memory=True)


def run_small_loss(noise_rate, noise_type, n_epochs=30):
    torch.manual_seed(SEED); np.random.seed(SEED)
    ds         = NoisyDataset(clean_trainset, noise_rate, noise_type)
    full_loader = make_loader(ds)
    model       = get_resnet18()
    opt, sched  = make_optimizer_scheduler(model, n_epochs=n_epochs)
    criterion   = nn.CrossEntropyLoss()
    history     = {'train_loss':[], 'train_acc':[], 'test_acc':[],
                   'keep_ratio':[]}
    warmup_end  = n_epochs // 2   # on atteint le ratio final à mi-training
    clean_loader = full_loader    # commence avec tous les samples

    for ep in range(n_epochs):
        # keep_ratio décroît de 1.0 → (1 - noise_rate)
        progress   = min(ep / max(1, warmup_end), 1.0)
        keep_ratio = 1.0 - noise_rate * progress

        # Recalculer le subset tous les 5 epochs
        if ep % 5 == 0:
            clean_loader = select_clean_samples(
                model, full_loader, ds, keep_ratio)

        model.train(); loss_sum = correct = total = 0
        for imgs, lbls, _ in clean_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward(); opt.step()
            loss_sum += loss.item()
            correct  += out.argmax(1).eq(lbls).sum().item()
            total    += lbls.size(0)
        sched.step()
        te_acc = evaluate(model, test_loader)
        history['train_loss'].append(loss_sum/len(clean_loader))
        history['train_acc'].append(100.*correct/total)
        history['test_acc'].append(te_acc)
        history['keep_ratio'].append(keep_ratio)
        if (ep+1) % 10 == 0:
            print(f'  Ep {ep+1:3d} | '
                  f'Keep {keep_ratio:.2f} | '
                  f'Loss {loss_sum/len(clean_loader):.3f} | '
                  f'Train {100.*correct/total:.1f}% | '
                  f'Test {te_acc:.1f}%')

    history['best_test_acc'] = max(history['test_acc'])
    return history

print('✅ Small-Loss Selection défini')

In [ ]:
results_small_loss = {}
t0 = time.time()

print('🚀 SMALL-LOSS SELECTION (Co-Teaching simplifié)')
print('=' * 55)

for nr, nt in CONFIGS:
    key = f'{nt}_{nr}'
    print(f'\n▶ {nt:<11} | ε = {nr}')
    hist = run_small_loss(nr, nt, n_epochs=30)
    results_small_loss[key] = hist
    print(f'   ✅ Best test acc : {hist["best_test_acc"]:.1f}%')

print(f'\n✅ Small-Loss terminé en {(time.time()-t0)/60:.1f} min')
with open('results_small_loss.pkl', 'wb') as f:
    pickle.dump(results_small_loss, f)
print('💾 results_small_loss.pkl sauvegardé')

---
## 📊 Section 4 — Aperçu rapide des résultats

In [ ]:
# Charger les résultats baseline de NB2
try:
    with open('/kaggle/input/nb2-outputs/results_baseline.pkl','rb') as f:
        results_baseline = pickle.load(f)
    print('✅ results_baseline.pkl chargé depuis /kaggle/input')
except FileNotFoundError:
    try:
        with open('results_baseline.pkl','rb') as f:
            results_baseline = pickle.load(f)
        print('✅ results_baseline.pkl chargé localement')
    except FileNotFoundError:
        print('⚠️  results_baseline.pkl non trouvé — run NB2 first')
        results_baseline = {}

# Tableau de comparaison rapide
import pandas as pd
rows = []
for nr, nt in CONFIGS:
    key = f'{nt}_{nr}'
    row = {'Noise Type': nt.capitalize(), 'ε': f'{int(nr*100)}%'}
    for name, rd in [
        ('Baseline',      results_baseline),
        ('LabelSmooth',   results_smoothing),
        ('Reweighting',   results_reweighting),
        ('SmallLoss',     results_small_loss),
    ]:
        row[name] = f"{rd[key]['best_test_acc']:.1f}%" if key in rd else '—'
    rows.append(row)

df = pd.DataFrame(rows)
print('\n' + '='*68)
print('APERÇU — Best Test Accuracy par méthode')
print('='*68)
print(df.to_string(index=False))
print('='*68)
print()
print('📊 Analyse approfondie → lancer NB4_analysis.ipynb')
print('   (charge tous les .pkl et génère les graphiques finaux)')